# Outlier：美国各州三个月后失业率预测
这份 Notebook 是组员理解、复现代码的入口，不是可提交的报告正文。按顺序运行；默认复用仓库内的 BLS 快照和已完成的模型回执。

**问题：** 当参考月 t 的州级数据已公布，能否用当前与历史信息预测同一个州参考月 t+3 的失业率，并超过“未来等于当前”的 persistence baseline？

**先记住：** 使用季调后的、当前已修订历史数据，结果只支持 retrospective evaluation。它没有重建历史发布版本。阅读 README 和 data/data_dictionary.md 后再解释模型结果。


## 1. 环境与路径
从仓库根目录或 src/ 目录打开。需要 requirements.txt 的五个包；此处不自动安装依赖，也不读取个人机器的绝对路径。


In [1]:
from pathlib import Path
import sys
import json
import importlib.metadata
import platform
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "src/bls_pipeline.py").exists():
    ROOT = ROOT.parent
assert (ROOT / "src/bls_pipeline.py").exists(), "请从仓库根目录或 src/ 目录运行"
sys.path.insert(0, str(ROOT / "src"))
from bls_pipeline import (collect, prepare, request_plan, series_mapping,
                          API_V1, FEATURES, NUMERIC_FEATURES, TARGET, month_cv)
from train_models import run_models
from make_eda import make_figures

print("Python:", platform.python_version())
print({p: importlib.metadata.version(p) for p in
       ["numpy", "pandas", "scikit-learn", "matplotlib", "joblib"]})


Python: 3.11.15
{'numpy': '2.4.4', 'pandas': '3.0.2', 'scikit-learn': '1.9.0', 'matplotlib': '3.11.0', 'joblib': '1.5.3'}


## 2. API request 是怎样组成的
BLS 的季调州级失业率 series ID 示例：LASST440000000000003。LA=LAUS，S=季调，ST44…=Rhode Island，03=失业率；04/05/06 分别是失业人数/就业人数/劳动力。

下面展示实际 POST 的 endpoint 和 JSON 结构。真正的 urllib.request.Request / urlopen、重试、批次和保存逻辑在 bls_pipeline.py 的 fetch_response() 和 collect() 中。这里不额外发送示例请求，避免重复消耗配额。


In [2]:
mapping = series_mapping(ROOT)
plan = request_plan(list(mapping), 2014, 2024, registered=False)
example_request = {"seriesid": ["LASST440000000000003"],
                   "startyear": "2014", "endyear": "2023"}
print("POST", API_V1)
print(json.dumps(example_request))
print(f"{len(mapping)} series; {len(plan)} anonymous requests for the complete collection")
assert all(len(batch["seriesid"]) <= 25 for batch in plan)


POST https://api.bls.gov/publicAPI/v1/timeseries/data/
{"seriesid": ["LASST440000000000003"], "startyear": "2014", "endyear": "2023"}
204 series; 18 anonymous requests for the complete collection


## 3. 采集并核对原始快照
默认 collect() 验证并复用 20261014T164700258612Z 快照，没有额外 HTTP 请求。首次空仓库运行才会真正请求 API。要有意识地获取新快照，用命令行 collect --refresh；未注册限 25 requests/day，本项目一次需 18。

每个响应保存查询、UTC 时间、原始响应；manifest 保存 SHA-256。M13 是年度平均，不能当第 13 个月。


In [3]:
snapshot = collect(ROOT)
manifest = json.loads((snapshot / "manifest.json").read_text())
print({"snapshot": snapshot.name, "complete": manifest["complete"],
       "batches": len(manifest["responses"]), "registered": manifest["registered"],
       "vintage_policy": manifest["vintage_policy"]})


Reusing frozen snapshot: data/raw/bls/20261014T164700258612Z
{'snapshot': '20261014T164700258612Z', 'complete': True, 'batches': 18, 'registered': False, 'vintage_policy': 'Currently revised data; not historical real-time vintages.'}


## 4. 数据质量、日历特征与标签
prepare() 先查重复、缺失、数值范围和 E+U=L，再按州建立连续月份日历。目标是 shift(-3)，历史是 shift(1/3/6/12)；先补齐日历才能确保月份缺口不会变成“隔三行”。

2014 年只提供 lag 历史。2024 年最后三个月没有 t+3 标签。边界月份有标签但会按可用时间排除，不能随机拆分行。


In [4]:
features = prepare(ROOT, snapshot)
panel = pd.read_csv(ROOT / "data/processed/state_month_panel.csv")
model = pd.read_csv(ROOT / "data/processed/model_table.csv")
quality = json.loads((ROOT / "results/data_quality.json").read_text())
print("Panel shape:", panel.shape, "Feature shape:", features.shape, "Labeled shape:", model.shape)
print("Quality passed:", quality["passed"], "Missing:", quality["missing_by_measure"])
sample = model[model.state == "Rhode Island"].head(4)
print(sample[["state", "month", "unemployment_rate", "ur_change_3m",
              "target_month", TARGET, "split"]].to_string(index=False))


{
  "quality_passed": true,
  "panel_rows": 6732,
  "labeled_rows": 5967,
  "split_rows": {
    "train": 2907,
    "test": 1683,
    "excluded": 1071,
    "validation": 1071
  }
}
Panel shape: (6732, 6) Feature shape: (6732, 24) Labeled shape: (5967, 24)
Quality passed: True Missing: {'unemployment_rate': 0, 'unemployed': 0, 'employed': 0, 'labor_force': 0}
       state   month  unemployment_rate  ur_change_3m target_month  target_unemployment_rate split
Rhode Island 2015-01                6.5          -0.6      2015-04                       6.3 train
Rhode Island 2015-02                6.4          -0.5      2015-05                       6.2 train
Rhode Island 2015-03                6.4          -0.3      2015-06                       6.0 train
Rhode Island 2015-04                6.3          -0.2      2015-07                       5.9 train


## 5. Train / validation / test 与三个月隔离
所有州在同一个参考月属于同一个集合。训练标签必须早于 validation 首月；validation 标签必须早于 test 首月。训练内的 CV 先切 unique months，再映射回各州行，所以 gap=3 是三个月，不是三行。

模型输入 16 列；target、target_month、split、排除原因都不会作为 predictor。StandardScaler 和 OneHotEncoder 在 sklearn Pipeline 内只拟合各自 training fold。


In [5]:
split_receipt = json.loads((ROOT / "results/split_summary.json").read_text())
print(pd.DataFrame(split_receipt["boundaries"]).T.to_string())
print("Excluded:", split_receipt["excluded_reasons"])
print("Predictors:", FEATURES)
train = model[model.split == "train"].reset_index(drop=True)
validation = model[model.split == "validation"]
test = model[model.split == "test"]
assert train.target_month.max() < validation.month.min()
assert validation.target_month.max() < test.month.min()
assert model.groupby("month").split.nunique().eq(1).all()
for index, (a, b) in enumerate(month_cv(train), 1):
    earlier, later = train.iloc[a], train.iloc[b]
    assert earlier.target_month.max() < later.month.min()
    print(index, "train label ends:", earlier.target_month.max(),
          "CV validation begins:", later.month.min())


            rows origin_min origin_max target_max
test        1683    2022-01    2024-09    2024-12
train       2907    2015-01    2019-09    2019-12
validation  1071    2020-01    2021-09    2021-12
Excluded: {'warmup_before_study_start': 612, 'label_crosses_next_block': 306, 'target_unavailable': 153}
Predictors: ['state', 'unemployment_rate', 'ur_change_1m', 'ur_change_3m', 'ur_change_6m', 'ur_change_12m', 'employed_growth_1m', 'employed_growth_3m', 'employed_growth_6m', 'employed_growth_12m', 'labor_force_growth_1m', 'labor_force_growth_3m', 'labor_force_growth_6m', 'labor_force_growth_12m', 'month_sin', 'month_cos']
1 train label ends: 2016-09 CV validation begins: 2016-10
2 train label ends: 2017-09 CV validation begins: 2017-10
3 train label ends: 2018-09 CV validation begins: 2018-10


## 6. 三张 EDA 图与对应 CSV
这些图涵盖全时段；test 也已通过 EDA 被观察。州均值不是官方全国失业率；2020–2021 的分组按 origin month 定义。图用于团队讨论，最终报告的解释与 caption 由团队自己写。


In [6]:
eda = make_figures(ROOT)
print(json.dumps(eda, indent=2))


Saved three exploratory figures and their source tables to figures
{
  "rows": 5967,
  "unemployment_rate_range_percent": [
    1.7,
    30.5
  ],
  "three_month_change_range_percentage_points": [
    -13.8,
    26.3
  ],
  "notes": [
    "Exploratory EDA includes all labeled periods, including the eventual test period; the test period is not pristine for later adaptive research.",
    "State lines use common axes. Equal-state mean is not a labor-force-weighted national estimate.",
    "Period grouping uses forecast origin month, not target month; no causal interpretation."
  ]
}


![State trends](../figures/01_unemployment_trends.png)

![Current vs future](../figures/02_current_vs_future.png)

![Three-month changes](../figures/03_three_month_changes.png)


## 7. Baseline、模型选择与已经观察过的 test
Persistence 直接预测 u(t+3)=u(t)。Ridge 与 Random Forest 在训练内调参，然后按 validation MAE 选一个模型（baseline 也参与选择），锁定后才评分 test。

下面默认读取相同输入/源码下已有回执，不再次评分。若修改过数据或模型源码，会要求显式 --rerun，旧回执会归档。重跑不是新的 holdout。当前已有结果中，Random Forest 被 validation 选中，但 test 不如 persistence；不能据此继续在同一个 test 上挑模型并声称独立验证。


In [7]:
receipt = run_models(ROOT)
validation_metrics = pd.read_csv(ROOT / "results/validation_metrics.csv")
print("Validation:")
print(validation_metrics.to_string(index=False))
print("Test (percentage points):")
print(pd.DataFrame(receipt["test_metrics"]).T.to_string())
print("Selected by validation:", receipt["selected_model"])
print("Scope:", receipt["scope"])


Reusing completed exploratory results; test has already been observed.
{
  "persistence": {
    "mae_percentage_points": 0.14105763517528222,
    "rmse_percentage_points": 0.19763906556566904
  },
  "random_forest": {
    "mae_percentage_points": 0.3017034017924023,
    "rmse_percentage_points": 0.5801895356519232
  }
}
Validation:
        model  mae_percentage_points  rmse_percentage_points
random_forest               1.811014                3.587864
  persistence               2.293371                3.891022
        ridge               5.961766               12.794929
Test (percentage points):
               mae_percentage_points  rmse_percentage_points
persistence                 0.141058                0.197639
random_forest               0.301703                0.580190
Selected by validation: random_forest
Scope: Exploratory retrospective evaluation on currently revised BLS data; not real-time vintage evaluation, causal evidence, or deployment validation.


## 8. 组员现在可以接手的任务
1. 对照 data dictionary 检查变量定义、百分比/百分点和发布时点，理解原始 API 返回。
2. 读 BLS 的季调/修订说明和失业率预测相关文献；记录“当前修订版本回测”与“真实实时预测”的差别。
3. 用已经导出的图和验证结果讨论哪些描述有证据，哪些只是解释假说；不要把相关性写成因果。
4. 自己撰写 midterm：1500 words 内、至少三张图、来源、每列说明和潜在用途。提交前核对官方 rubric。

**理解检查：** 为什么不能随机 train_test_split？为什么 2024-10 的标签不能填当前失业率？为什么 4%→5% 是 1 percentage point 而不是 1% relative growth？为什么失业人数和劳动力人数的比值已基本给出了失业率？

无需为回答这些问题重新训练或观察新的测试结果。
